In [1]:
# standard
from pathlib import Path
import os

# 3rd party
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# BIRD 
from comb.behavior_ophys_dataset import BehaviorOphysDataset, BehaviorMultiplaneOphysDataset
import lamf_analysis.ophys.stim_response as sr
from aind_ophys_data_access import file_handling

from lamf_analysis.ophys.dff_utils import plot_dff_traces_examples, plot_population_dff

# notebook dev
%load_ext autoreload
%autoreload 2
%matplotlib inline


import sys
sys.path.append('/root/capsule/code')
import capsule_utils


['/comb/src/comb', '/opt/conda/lib/python39.zip', '/opt/conda/lib/python3.9', '/opt/conda/lib/python3.9/lib-dynload', '', '/opt/conda/lib/python3.9/site-packages', '/lamf-analysis/src', '/aind-codeocean-api/src', '/brain_observatory_utilities_ophys', '/comb/src', '/aind-multiplane-ophys-qc/src', '/aind-ophys-data-access/src', '/root/capsule/code']


In [49]:
raw_path = Path("/root/capsule/data/multiplane-ophys_741866_2024-09-18_09-46-53")
processed_path = Path("/root/capsule/data/multiplane-ophys_741866_2024-09-18_09-46-53_processed_2024-11-19_17-31-51/VISp_0")

bod = BehaviorOphysDataset(plane_folder_path=processed_path,
                           raw_folder_path=raw_path,
                           project_code="LearningmFISHTask1A",
                           pipeline_version='v6-from_lims')

INFO:root:Trying to find sync file using platform json for /root/capsule/data/multiplane-ophys_741866_2024-09-18_09-46-53
INFO:comb.file_handling:Sync file found in /root/capsule/data/multiplane-ophys_741866_2024-09-18_09-46-53/behavior/1395311538_20240918T094635.h5


{'plane_group_count': 1, 'plane_group_index': 0, 'split_json_scanfield_z': 80}
<KeysViewHDF5 ['baseline', 'data', 'noise', 'skewness']>


INFO:root:Trying to find sync file using platform json for /root/capsule/data/multiplane-ophys_741866_2024-09-18_09-46-53
INFO:comb.file_handling:Sync file found in /root/capsule/data/multiplane-ophys_741866_2024-09-18_09-46-53/behavior/1395311538_20240918T094635.h5


In [60]:
bod.dff_traces

,dff,baseline,noise,skewness
cell_specimen_id,,,,
0,"[-0.022535186, 0.18213569, -0.0045963423, 0.45...","[326.07968, 326.07968, 325.62936, 325.62936, 3...",52.816521,1.053979
1,"[0.1885793, 0.045886658, -0.08124229, 0.006901...","[893.2957, 893.2957, 893.1173, 893.1173, 893.1...",88.001999,1.228537
2,"[0.070005104, 0.04911207, -0.10392792, 0.04590...","[672.91064, 672.91064, 672.91064, 672.91064, 6...",65.663277,1.209525
3,"[-0.23127267, -0.1169827, -0.035727214, -0.202...","[940.17017, 940.17017, 940.17017, 940.17017, 9...",58.24691,2.475454
4,"[0.19745493, 0.19106866, 0.25581324, 0.1844987...","[1721.7561, 1721.7561, 1721.7561, 1721.7561, 1...",66.140778,1.816293
...,...,...,...,...
405,"[-0.054635566, -0.092405364, 0.15635377, -0.05...","[127.55054, 127.55054, 127.55054, 127.92395, 1...",55.56797,1.803988
406,"[0.07686475, -0.2475522, 0.29095545, 0.4563954...","[423.16324, 422.57846, 422.57846, 422.57846, 4...",53.009373,0.370854
407,"[0.08817874, -0.03556108, -0.08262107, -0.0237...","[489.32025, 489.32025, 489.32025, 488.77972, 4...",67.877182,1.171191


In [38]:
np.array([1,2,3])

array([1, 2, 3])

In [37]:
roi_table = pd.DataFrame(index=range(10))
roi_table.loc[2, "lol"]= np.array([1,2,3])
roi_table

ValueError: setting an array element with a sequence.

# Dev area: individual considerations for v6

1. plane sort index (_infer_plane_sort_index(self), _add_plane_order_index(self))
2. Plane group and indexes for sync parsing
    + continue using mesoscope file splitting json
    + TODO: later can remove parsing the mesoscope file splitting json
        +  split_dict['plane_group_index'] and split_dict['plane_group_count'] should be in session json
3. Extraction capsule
    

In [ ]:
# just copied code, things look fine in comb for now.


    # def _infer_plane_sort_index(self):
    #     """The names of the plane folders, sorted, determine the order of the planes
        
    #     In some cases this is used for container assignment.
        
    #     Returns
    #     -------
    #     plane_folder_index_map : dict
    #     """
        
    #     if self.pipeline_version == 'v6':
            
    #         # TODO: getting plane folders and validating can be elsehwere
    #         # TODO: look up from brain areas?
    #         valid_prefix = ["VISp"]
            
    #         processed_path = self.metadata["plane_path"].parent
    #         plane_folders = [f for f in processed_path.iterdir() if f.is_dir()]
            
    #         assert all([f.name.split('_')[1].isdigit() for f in plane_folders]), "Plane folders are not named as expected"
    #         assert all([f.name.split('_')[0] in valid_prefix for f in plane_folders]), "Plane folders are not named as expected"
            
    #         plane_folder_index_map = {f.name: int(f.name.split('_')[1]) for f in plane_folders}

    #     else:
    #         # generally else = v4, specifically for saffron sessions
            
    #         processed_path = self.metadata["plane_path"].parent
    #         plane_folders = [f for f in processed_path.iterdir() if f.is_dir()]
            
    #         # assert 8 folders
    #         assert len(plane_folders) == 8, f"Expected 8 plane folders, found {len(plane_folders)}"
    #         plane_folders = sorted(plane_folders, key=lambda x: x.name)
    #         plane_folder_index_map = {plane_folder.name: i for i, plane_folder in enumerate(plane_folders)}
        
    #     return plane_folder_index_map

## Directly load session json to look at ophys index

In [26]:
 from aind_ophys_data_access import metadata
 json_dicts = metadata.load_metadata_json_files(raw_path)
 ophys_fovs_dict = metadata.extract_ophys_fovs(json_dicts["session"], index_key_only = False)
 ophys_fovs_dict

{'VISp_0': {'coupled_fov_index': 0,
  'fov_coordinate_ap': '1.5',
  'fov_coordinate_ml': '1.5',
  'fov_coordinate_unit': 'micrometer',
  'fov_height': 512,
  'fov_reference': 'Bregma',
  'fov_scale_factor': '0.78',
  'fov_scale_factor_unit': 'um/pixel',
  'fov_size_unit': 'pixel',
  'fov_width': 512,
  'frame_rate': '37.92',
  'frame_rate_unit': 'hertz',
  'imaging_depth': 175,
  'imaging_depth_unit': 'micrometer',
  'index': 0,
  'magnification': '16x',
  'notes': None,
  'power': '33.0',
  'power_ratio': '36.0',
  'power_unit': 'percent',
  'scanfield_z': 80,
  'scanfield_z_unit': 'micrometer',
  'scanimage_roi_index': 0,
  'targeted_structure': 'VISp'},
 'VISp_1': {'coupled_fov_index': 0,
  'fov_coordinate_ap': '1.5',
  'fov_coordinate_ml': '1.5',
  'fov_coordinate_unit': 'micrometer',
  'fov_height': 512,
  'fov_reference': 'Bregma',
  'fov_scale_factor': '0.78',
  'fov_scale_factor_unit': 'um/pixel',
  'fov_size_unit': 'pixel',
  'fov_width': 512,
  'frame_rate': '37.92',
  'frame

Here is the dict keys for v4 comb, need to make them plane names now that those are valid.


{0: {'coupled_fov_index': 0,
  'fov_coordinate_ap': '1.5',
  'fov_coordinate_ml': '1.5',
  'fov_coordinate_unit': 'micrometer',
  'fov_height': 512,
  'fov_reference': 'Bregma',
  'fov_scale_factor': '0.78',
  'fov_scale_factor_unit': 'um/pixel',
  'fov_size_unit': 'pixel',
  'fov_width': 512,
  'frame_rate': '37.92',
  'frame_rate_unit': 'hertz',
  'imaging_depth': 175,
  'imaging_depth_unit': 'micrometer',
  'index': 0,
  'magnification': '16x',
  'notes': None,
  'power': '33.0',
  'power_ratio': '36.0',
  'power_unit': 'percent',
  'scanfield_z': 80,
  'scanfield_z_unit': 'micrometer',
  'scanimage_roi_index': 0,
  'targeted_structure': 'VISp'},
 1: {'coupled_fov_index': 0,
  'fov_coordinate_ap': '1.5',
  'fov_coordinate_ml': '1.5',
  'fov_coordinate_unit': 'micrometer',
  'fov_height': 512,
  'fov_reference': 'Bregma',
  'fov_scale_factor': '0.78',
  'fov_scale_factor_unit': 'um/pixel',
  'fov_size_unit': 'pixel',
  'fov_width': 512,
  'frame_rate': '37.92',
  'frame_rate_unit': 'hertz',
  'imaging_depth': 275,
  'imaging_depth_unit': 'micrometer',
  'index': 1,
  'magnification': '16x',
  'notes': None,
  'power': '33.0',
  'power_ratio': '36.0',
  'power_unit': 'percent',
  'scanfield_z': 165,
  'scanfield_z_unit': 'micrometer',
  'scanimage_roi_index': 0,
  'targeted_structure': 'VISp'}}

# Extraction

In [3]:
from comb.file_handling import load_sparse_array, load_generic_group, load_signals



In [10]:
extraction_h5 = "/root/capsule/data/multiplane-ophys_741866_2024-09-18_09-46-53_processed_2024-11-19_17-31-51/VISp_0/extraction/VISp_0_extraction.h5"

pixel_masks = load_sparse_array(extraction_h5)
pixel_masks.shape

(410, 512, 512)

In [23]:
type(roi_names[0])

int

In [12]:
roi_shape = load_generic_group(extraction_h5, h5_group="rois", h5_key="shape")

h5 /root/capsule/data/multiplane-ophys_741866_2024-09-18_09-46-53_processed_2024-11-19_17-31-51/VISp_0/extraction/VISp_0_extraction.h5


array([410, 512, 512], dtype=int16)

In [25]:
load_generic_group(extraction_h5, h5_group="rois", h5_key="data")

h5 /root/capsule/data/multiplane-ophys_741866_2024-09-18_09-46-53_processed_2024-11-19_17-31-51/VISp_0/extraction/VISp_0_extraction.h5


(89387,)

In [18]:
raw_traces, roi_names = load_signals(extraction_h5, h5_group="traces", h5_key="roi")
nueropil_traces, roi_names = load_signals(extraction_h5, h5_group="traces", h5_key="neuropil")
corrected_traces, roi_names = load_signals(extraction_h5, h5_group="traces", h5_key="corrected")

# dfof_traces, roi_names = load_signals(plane_files['dff_h5'], ps, h5_key = "data")
#event_traces, roi_names = load_signals(plane_files['events_oasis_h5'], ps, h5_key='events',)

In [20]:
corrected_traces.shape

(410, 170918)